# NegotiateEnv Training - WebSocket - OpenEnv Hackathon

**Team**: Mayuka Reddy & Kushal Adhyaru  
**Environment**: https://kushaladhyaru-negotiate-env.hf.space  
**Repository**: https://github.com/MadhaviSG/openEnv-negotiateEnv

---

## Configuration

- **Protocol**: WebSocket (proper OpenEnv)
- **Baseline episodes**: 200
- **Training episodes**: 500
- **Method**: HuggingFace TRL with LoRA
- **Baseline**: 0.4833
- **Target**: 0.55-0.60 (15-25% improvement)

**Advantages**:
- Proper session management
- Reliable connection
- OpenEnv compliant
- Works with deployed Space

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 2. Load Repository

In [ ]:
import os

if os.path.exists('openEnv-negotiateEnv.zip'):
    print("Extracting uploaded repository...")
    !unzip -q openEnv-negotiateEnv.zip
    %cd openEnv-negotiateEnv
elif os.path.exists('openEnv-negotiateEnv'):
    %cd openEnv-negotiateEnv
else:
    !git clone https://github.com/MadhaviSG/openEnv-negotiateEnv.git
    %cd openEnv-negotiateEnv

!ls -la train_websocket_trl.py

## 3. Install Dependencies

In [ ]:
!pip install -q -e .
!pip install -q transformers accelerate peft datasets trl websockets matplotlib numpy

print("\nAll dependencies installed!")

## 4. Test WebSocket Connection

In [ ]:
import asyncio
import websockets
import json

ENV_URL = "wss://kushaladhyaru-negotiate-env.hf.space"

async def test_websocket():
    print(f"Testing WebSocket connection to {ENV_URL}...\n")
    print("="*60)
    
    try:
        async with websockets.connect(ENV_URL, timeout=10) as websocket:
            print("[OK] WebSocket connected!")
            
            # Test reset
            await websocket.send(json.dumps({"type": "reset"}))
            response = await websocket.recv()
            data = json.loads(response)
            print("[OK] Reset successful!")
            print(f"   Scenario: {data.get('observation', {}).get('context', '')[:60]}...")
            
            # Test step
            action = {
                "action_type": "probe",
                "price_per_seat": 0.0,
                "contract_length": 0.0,
                "annual_increase_cap": 0.0,
                "message": "Tell me more"
            }
            await websocket.send(json.dumps({"type": "step", "action": action}))
            response = await websocket.recv()
            data = json.loads(response)
            print("[OK] Step successful!")
            print(f"   Done: {data.get('observation', {}).get('done')}")
            print(f"   Reward: {data.get('observation', {}).get('reward')}")
            
            print("\n" + "="*60)
            print("[SUCCESS] WebSocket is working!")
            print("[SUCCESS] Ready for training!")
            print("="*60)
            
    except Exception as e:
        print(f"\n[ERROR] WebSocket test failed: {e}")
        print("\nTroubleshooting:")
        print("1. Check your Space is running")
        print("2. Visit: https://kushaladhyaru-negotiate-env.hf.space")
        print("3. Wait for it to wake up (30 seconds)")
        print("4. Run this cell again")

await test_websocket()

## 5. Run Training (500 Episodes)

**Training Configuration:**
- Model: Qwen/Qwen2.5-1.5B-Instruct
- Protocol: WebSocket
- Episodes: 500
- Time on H100: ~25-35 minutes
- Time on T4: ~2-3 hours

**Baseline: 0.4833**  
**Target: 0.55-0.60**

In [ ]:
ENV_URL = "https://kushaladhyaru-negotiate-env.hf.space"

print("Starting WebSocket training...")
print(f"Environment: {ENV_URL}")
print("Episodes: 500\n")
print("Expected time:")
print("  - H100: ~25-35 minutes")
print("  - T4: ~2-3 hours\n")
print("="*60)

!python train_websocket_trl.py \
    --model-id Qwen/Qwen2.5-1.5B-Instruct \
    --env-url {ENV_URL} \
    --output-dir negotiate-trl-output \
    --num-episodes 500 \
    --max-turns 10

print("\n" + "="*60)
print("Training complete!")
print("Model saved to: negotiate-trl-output/")
print("="*60)

## 6. Save Model to HuggingFace Hub

In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('Huggingface_Token')
    login(token=hf_token)
    print("Successfully logged in!")
except Exception as e:
    print(f"[ERROR] {e}")
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
import os

api = HfApi()
output_dir = "negotiate-trl-output"
repo_id = "KushalAdhyaru/negotiate-env-qwen-websocket-500ep"

if os.path.exists(output_dir):
    print(f"Uploading to {repo_id}...\n")
    
    try:
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            repo_type="model",
        )
        print("\nModel uploaded successfully!")
        print(f"View at: https://huggingface.co/{repo_id}")
    except Exception as e:
        print(f"\n[ERROR] {e}")
else:
    print(f"[ERROR] Directory not found: {output_dir}")

## Final Summary

### Hackathon Submission:

- Environment: https://kushaladhyaru-negotiate-env.hf.space
- Dataset: https://huggingface.co/datasets/mayukareddy/SyntheticSaasDataset
- Model: https://huggingface.co/KushalAdhyaru/negotiate-env-qwen-websocket-500ep
- Code: https://github.com/MadhaviSG/openEnv-negotiateEnv
- Protocol: WebSocket (OpenEnv compliant)

**Good luck!**